# Lab 4 · Exercise B — HistFactory on CMS $H\\to 4\\ell$
CMS Open Data (arXiv:1207.7235): four-lepton mass, **real data** and simulated templates.
Two channels: a **signal region** (70–181 GeV, 37 bins) and a **control region** (one bin, $m_{4\\ell}>181$ GeV) that measures the $ZZ$ normalisation:
the same idea as the on/off problem, with histograms.

In [ ]:
import ROOT
import numpy as np

ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name
import os

In [ ]:
D = "../data/cms_m4l/"
def load(name):
    return np.loadtxt(D + "_cms_higgs_%s.txt" % name)[:, 1]

data_v = load("data")
tmpl_v = {"signal": load("HZZto4l"), "ZZ": load("ZZto4l"), "DY": load("DYto4l"), "ttbar": load("TTbarto4l")}

# entries: 0 = empty underflow, 1..37 = 70-181 GeV (3 GeV bins), 38 = overflow: m4l > 181 GeV
SR, CR = slice(1, 38), 38

def th1(name, vals, nbins, lo, hi):
    """Histogram that HistFactory may keep: detach from the file and from Python ownership."""
    h = ROOT.TH1D(name, name, nbins, lo, hi)
    for i, v in enumerate(vals):
        h.SetBinContent(i + 1, float(v))
    h.SetDirectory(0)
    ROOT.SetOwnership(h, False)
    return h

print("data: SR %d events, CR %d events" % (data_v[SR].sum(), data_v[CR]))
print({k: round(float(v[SR].sum()), 2) for k, v in tmpl_v.items()}, "ZZ in CR:", round(float(tmpl_v["ZZ"][CR]), 1))

In [ ]:
check("data in the signal region: 102 events", int(data_v[SR].sum()) == 102)
check("data in the control region: 176 events", int(data_v[CR]) == 176)

## B1 · Build the Measurement
Complete `build_meas`: the signal carries the `NormFactor` `mu` (POI); $ZZ$ carries `kZZ`. Because both channels use the **same name**,
the parameter is shared. $H\\to4\\ell$ templates enter the control region as well (their yield there is tiny).

In [ ]:
ROOT.gInterpreter.Declare('#include "RooStats/HistFactory/MakeModelAndMeasurementsFast.h"')
HF = ROOT.RooStats.HistFactory
os.makedirs("results", exist_ok=True)

def build_meas(use_cr=True, free_zz=True, lumi_err=0.0, tag="cms"):
    meas = HF.Measurement("m_" + tag, "CMS H->4l")
    meas.SetOutputFilePrefix("results/" + tag)
    meas.SetPOI("mu")
    meas.SetLumi(1.0)
    meas.SetLumiRelErr(lumi_err if lumi_err > 0 else 0.01)
    if lumi_err == 0:
        meas.AddConstantParam("Lumi")
    meas.SetExportOnly(True)          # we fit ourselves

    regions = [("SR", SR, 37, 70.0, 181.0)] + ([("CR", CR, 1, 181.0, 600.0)] if use_cr else [])
    for rname, sl, nb, lo, hi in regions:
        ch = HF.Channel(rname)
        vals = lambda arr: np.atleast_1d(arr[sl])
        ch.SetData(th1("data_" + rname, vals(data_v), nb, lo, hi))
        for sname, v in tmpl_v.items():
            s = HF.Sample(sname)
            s.SetHisto(th1(sname + "_" + rname, vals(v), nb, lo, hi))
            s.SetNormalizeByTheory(lumi_err > 0 and sname != "ZZ")
            # TODO: signal gets  s.AddNormFactor("mu", 1, 0, 5)
            # TODO: ZZ gets      s.AddNormFactor("kZZ", 1, 0.1, 3)   (only if free_zz): shared by SR and CR
            ch.AddSample(s)
        meas.AddChannel(ch)
    return meas

In [ ]:
_n = [0]
def run(use_cr=True, free_zz=True, lumi_err=0.0):
    """Build, fit, and analyse with RooStats. Returns a dict of results."""
    _n[0] += 1
    meas = build_meas(use_cr, free_zz, lumi_err, tag="cms%d" % _n[0])
    ws = HF.MakeModelAndMeasurementFast(meas)
    mc = ws.obj("ModelConfig")
    pdf, data = mc.GetPdf(), ws.data("obsData")
    mu = ws.var("mu")
    res = pdf.fitTo(data, ROOT.RooFit.Save(), ROOT.RooFit.PrintLevel(-1))
    out = dict(ws=ws, mc=mc, pdf=pdf, data=data, res=res, mu=mu, mu_hat=mu.getVal(), mu_err=mu.getError(),
               k_hat=(ws.var("kZZ").getVal() if ws.var("kZZ") else 1.0))
    plc = ROOT.RooStats.ProfileLikelihoodCalculator(data, mc)
    plc.SetConfidenceLevel(0.683)
    iv = plc.GetInterval()
    out["lo"], out["hi"] = iv.LowerLimit(mu), iv.UpperLimit(mu)
    mu0 = ROOT.RooRealVar("mu", "mu", 0)
    plc.SetNullParameters(ROOT.RooArgSet(mu0))
    out["Z"] = plc.GetHypoTest().Significance()
    return out

In [ ]:
full = run(use_cr=True, free_zz=True)
print("mu = %.2f +- %.2f   kZZ = %.2f   68%% interval [%.2f, %.2f]   Z0 = %.2f" %
      (full["mu_hat"], full["mu_err"], full["k_hat"], full["lo"], full["hi"], full["Z"]))

In [ ]:
check("fit converged", full["res"].status() == 0)
check("mu close to 1", abs(full["mu_hat"] - 0.98) < 0.15)
check("kZZ below 1: data want less ZZ than the simulation", 0.8 < full["k_hat"] < 1.0)
check("evidence-level significance (3 to 4 sigma)", 3.0 < full["Z"] < 4.0)

## B2 · What does the control region buy us?
Repeat without the control region and with $k_{ZZ}$ fixed to 1.

In [ ]:
nocr = run(use_cr=False, free_zz=False)
print("no control region: mu = %.2f +- %.2f  [%.2f, %.2f]  Z0 = %.2f" % (nocr["mu_hat"], nocr["mu_err"], nocr["lo"], nocr["hi"], nocr["Z"]))
print("with control region: mu = %.2f +- %.2f  [%.2f, %.2f]  Z0 = %.2f" % (full["mu_hat"], full["mu_err"], full["lo"], full["hi"], full["Z"]))

In [ ]:
check("without the control region mu is lower (ZZ at its simulated size)", nocr["mu_hat"] < full["mu_hat"])
check("the control region raises the significance", full["Z"] > nocr["Z"])

## B3 · An illustrative luminosity uncertainty
Add a 5 % luminosity uncertainty (`lumi_err=0.05`): a global `OverallSys` on signal, DY and $t\\bar t$ ($ZZ$ is normalised by the control region).
**The 5 % is an illustrative size, not the CMS value.** Replace it with a number you can justify in your project.

In [ ]:
lum = run(use_cr=True, free_zz=True, lumi_err=0.05)
print("with 5%% luminosity: mu = %.2f +- %.2f  [%.2f, %.2f]  Z0 = %.2f" % (lum["mu_hat"], lum["mu_err"], lum["lo"], lum["hi"], lum["Z"]))

In [ ]:
check("a luminosity nuisance cannot shrink the uncertainty", lum["mu_err"] >= full["mu_err"] - 0.01)

## B4 · Anatomy of the likelihood
Slice vs profile of $\\mu$, and the correlation between $\\mu$ and $k_{ZZ}$ in the full model.

In [ ]:
pdf, data, mu, res = full["pdf"], full["data"], full["mu"], full["res"]
nll = pdf.createNLL(data)
frame = mu.frame(ROOT.RooFit.Range(0.0, 2.2))
# TODO: slice  (nll.plotOn(frame, ShiftToZero, black))  and  profile  (nll.createProfile(RooArgSet(mu)).plotOn(frame, ShiftToZero, red))
frame.SetMinimum(0); frame.SetMaximum(3)
c = ROOT.TCanvas("c", "c", 700, 500)
frame.Draw(); c.Draw()
corr = res.correlation("mu", "kZZ")
print("corr(mu, kZZ) = %.2f" % corr)

In [ ]:
check("mu and kZZ are not strongly correlated", abs(corr) < 0.5)

## Questions (3–4 lines each)
1. $\\hat k_{ZZ}$ is about 0.9. What does it say about the simulation? How does it change $\\hat\\mu$?
2. Why is $\\mu$ almost uncorrelated with $k_{ZZ}$, although both enter the signal region?
3. The data have 102 events in the signal region and the model predicts about 94. Is the excess at 125 GeV significant? What would a *look-elsewhere effect* change?
4. Which systematic uncertainties of the real CMS analysis are missing here? Which would matter most?